In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import random
import os
import json

load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=groq_api_key
)

In [2]:
def get_video_id(url):
    if "v=" in url:
        video_id = url.split("v=")[1]
        video_id = video_id.split("&")[0]
        return video_id
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
        video_id = video_id.split("?")[0]
        return video_id
    else:
        return None

def get_transcript(url):
    video_id = get_video_id(url)
    if not video_id:
        print("Invalid Youtube URL")
        return None
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return transcript
    except Exception as e:
        print("Error: Could not fetch transcript. The video may not have captions or may not exist.")
        return None

def chunk_transcript(transcript, chunk_duration=120):
    chunks = []
    current_text = ""
    chunk_start = transcript[0].start

    for segment in transcript:
        if segment.start - chunk_start >= chunk_duration:
            chunks.append({
                "text": current_text.strip(),
                "start": chunk_start,
                "end": segment.start
            })
            current_text = segment.text + " "
            chunk_start = segment.start
        else:
            current_text += segment.text + " "

    # Don't forget the last chunk that's still being built
    if current_text:
        chunks.append({
            "text": current_text.strip(),
            "start": chunk_start,
            "end": transcript[-1].start
        })

    return chunks


class TranscriptionAgent:
    def __init__(self):
        self.name = "Transcription Agent"
    
    def run(self, url):
        transcript = get_transcript(url)
        if not transcript:
            return None
        chunks = chunk_transcript(transcript)
        return chunks

In [3]:
class FilterAgent:
    def __init__(self, llm):
        self.name = "Filter Agent"
        self.llm = llm
    
    def run(self, chunks):
        educational_chunks = []
        for chunk in chunks:
            try:
                prompt = f"""Look at this transcript section and decide if it contains actual educational content or if it is filler content.

Filler content includes: introductions, outros, subscribe reminders, agenda overview, quiz segments, promotions, greetings, channel promotions, "like and share" requests, or any non-teaching content.

IMPORTANT: If the section contains ANY educational explanation, concepts, or teaching, even if mixed with casual talk, mark it as "educational".
Only mark as "skip" if the ENTIRE section is filler with NO educational value.

Transcript:
{chunk['text']}

Reply with ONLY one word: "educational" or "skip"
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip().lower()
                if content == "educational":
                    educational_chunks.append(chunk)
                else:
                    print(f"Skipping filler chunk at {chunk['start']:.0f}s")
            except Exception as e:
                print(f"Error filtering chunk: {e}")
                educational_chunks.append(chunk)
        
        print(f"Kept {len(educational_chunks)} out of {len(chunks)} chunks")
        return educational_chunks
        # For each chunk:
        #   Ask LLM if it's educational or filler
        #   If educational, keep it
        #   If filler, skip it
        # Return only educational chunks

In [4]:
class ConceptExtractionAgent:
    def __init__(self, llm):
        self.name = "Concept Extraction Agent"
        self.llm = llm
    
    def run(self, chunks):
        all_concepts = []
        
        for chunk in chunks:
            try:
                start_min = int(chunk["start"] // 60)
                start_sec = int(chunk["start"] % 60)
                end_min = int(chunk["end"] // 60)
                end_sec = int(chunk["end"] % 60)
                timestamp = f"{start_min}:{start_sec:02d} - {end_min}:{end_sec:02d}"
                
                prompt = f"""Extract the 2-3 most important concepts or topics from this transcript section.

Transcript:
{chunk['text']}

Return ONLY JSON in this format:
{{"concepts": ["concept 1", "concept 2", "concept 3"]}}
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:]
                if content.startswith("```"):
                    content = content[3:]
                if content.endswith("```"):
                    content = content[:-3]
                content = content.strip()
                
                concepts = json.loads(content)
                concepts["timestamp"] = timestamp
                all_concepts.append(concepts)
            except Exception as e:
                print(f"Skipping chunk: {e}")
                continue
        
        return all_concepts

In [5]:
class QuestionGenerationAgent:
    def __init__(self, llm):
        self.name = "Question Generation Agent"
        self.llm = llm

    def shuffle_options(self, question):
        options = list(question["options"].items())
        random.shuffle(options)
    
        old_correct = question["correct_answer"]
        correct_text = question["options"][old_correct]
    
        new_options = {}
        new_correct = None
        for i, (_, text) in enumerate(options):
            letter = ["A", "B", "C", "D"][i]
            new_options[letter] = text
            if text == correct_text:
                new_correct = letter
    
        question["options"] = new_options
        question["correct_answer"] = new_correct
        return question
    
    def run(self, concepts_list):
        all_questions = []
        
        for item in concepts_list:
            try:
                concepts = ", ".join(item["concepts"])
                timestamp = item["timestamp"]
                
                prompt = f"""Generate 1 multiple choice question that tests understanding of these concepts: {concepts}

The question should have:
- 4 options (A, B, C, D)
- The correct answer
- An explanation of why that answer is correct

Return ONLY JSON in this format:
{{
    "question": "the question text",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "correct_answer": "B",
    "explanation": "why this is correct"
}}
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:]
                if content.startswith("```"):
                    content = content[3:]
                if content.endswith("```"):
                    content = content[:-3]
                content = content.strip()
                
                question = json.loads(content)
                question = self.shuffle_options(question)
                question["timestamp"] = timestamp
                question["concepts"] = item["concepts"]
                all_questions.append(question)
            except Exception as e:
                print(f"Skipping question: {e}")
                continue
        
        return {"questions": all_questions}

In [6]:
class EvaluationAgent:
    def __init__(self, llm):
        self.name = "Evaluation Agent"
        self.llm = llm

    def deduplicate_topics(self, weak_topics):
        prompt = f"""Here is a list of topics:
{weak_topics}

Group similar or related topics together and give me a clean, 
deduplicated list. Combine topics that mean the same thing into 
one clear topic name.

Return ONLY JSON in this format:
{{"topics": ["topic 1", "topic 2", "topic 3"]}}
"""
        try:
            response = self.llm.invoke(prompt)
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
        
            result = json.loads(content)
            return result["topics"]
        except Exception as e:
            print(f"Deduplication failed: {e}")
            return weak_topics
    
    def run(self, quiz_data):
        score = 0
        weak_topics = []
        
        for i, q in enumerate(quiz_data["questions"]):
            print(f"\nQuestion {i+1}: {q['question']}")
            for option, text in q["options"].items():
                print(f"  {option}) {text}")
            user_answer = input("\nYour answer (A/B/C/D): ").upper()
            if user_answer == q['correct_answer']:
                score = score + 1
                print("Correct!")
            else:
                print("Incorrect!")
                weak_topics.extend(q['concepts'])
                weak_topics = list(set(weak_topics))
            print(f"Correct Answer: {q['correct_answer']}")
            print(f"Explanation: {q['explanation']}")
            print(f"Review in video at: {q['timestamp']}")
            
        print(f"\nFinal Score: {score}/{len(quiz_data['questions'])}")
        if weak_topics:
            weak_topics = self.deduplicate_topics(weak_topics)
        print(f"\nWeak Topics: {weak_topics}")

        return {
            "score": score,
            "total": len(quiz_data["questions"]),
            "weak_topics": weak_topics
        }

In [7]:
# Agent 1: Get transcript
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent 1 done: {len(chunks)} chunks")

# Agent 2: Extract concepts
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)
print(f"Agent 2 done: {len(concepts)} concept sets")

# Agent 3: Generate questions
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)
print(f"Agent 3 done: {len(quiz['questions'])} questions")

# Agent 4: Evaluate
agent4 = EvaluationAgent(llm)
results = agent4.run(quiz)

Agent 1 done: 10 chunks
Agent 2 done: 10 concept sets
Agent 3 done: 10 questions

Question 1: What type of machine learning model is commonly used for image recognition tasks, such as classifying objects in images?
  A) Support Vector Machine (SVM)
  B) Convolutional Neural Network (CNN)
  C) Decision Tree
  D) Linear Regression



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: Convolutional Neural Networks (CNNs) are a type of neural network specifically designed for image recognition tasks. They use convolutional and pooling layers to extract features from images, allowing them to learn complex patterns and relationships. This makes them well-suited for tasks such as object detection, image classification, and segmentation, which is why they are commonly used in image recognition applications.
Review in video at: 0:04 - 2:05

Question 2: What is the primary function of the activation function in a neural network neuron?
  A) To introduce non-linearity into the neuron's output
  B) To specify the type of data the network will process
  C) To connect multiple neurons together
  D) To determine the number of layers in the network



Your answer (A/B/C/D):  a


Correct!
Correct Answer: A
Explanation: The activation function is used to introduce non-linearity into the neuron's output, allowing the neural network to learn and represent more complex relationships between inputs and outputs. Without an activation function, the neuron's output would be a linear combination of its inputs, limiting the network's ability to learn and generalize.
Review in video at: 2:05 - 4:06

Question 3: In a neural network designed for digit recognition, which of the following best describes the primary function of the hidden layers?
  A) To store the network's weights and biases
  B) To output the probability of the input being each possible digit
  C) To accept user input and display the final recognized digit
  D) To process and transform the input data into a more abstract representation that aids in digit recognition


...

Incorrect!
Correct Answer: B
Explanation: The result of the matrix vector multiplication of a weight matrix W with dimensions 3x4 and an input vector x with dimensions 4x1 is a 3x1 vector, because the number of columns in the weight matrix matches the number of rows in the input vector, allowing for the multiplication to be performed, resulting in a vector with the same number of rows as the weight matrix.
Review in video at: 14:14 - 16:15

Question 9: When training a neural network, which activation function is more commonly used in hidden layers due to its ability to introduce non-linearity without saturating, unlike the sigmoid function?
  A) Softmax function
  B) Tanh function
  C) ReLU (Rectified Linear Unit) function
  D) Sigmoid function



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: C
Explanation: ReLU is more commonly used in hidden layers because it does not saturate like the sigmoid function, which can lead to vanishing gradients during backpropagation. ReLU is also computationally efficient and helps to avoid the vanishing gradient problem, making it a popular choice for deep neural networks.
Review in video at: 16:15 - 18:15

Question 10: What is the primary purpose of using the ReLU activation function in deep neural networks during the training process?
  A) To increase the risk of overfitting by adding more parameters
  B) To decrease the learning rate of the model
  C) To reduce the dimensionality of the input data
  D) To introduce non-linearity into the model, allowing it to learn more complex relationships



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: D
Explanation: ReLU (Rectified Linear Unit) is used to introduce non-linearity into the model. This is important because linear models can only learn linear relationships between inputs and outputs. By using ReLU, the model can learn more complex, non-linear relationships, which is essential for many real-world problems. ReLU also has the benefit of being computationally efficient and helping to avoid the vanishing gradient problem during backpropagation.
Review in video at: 18:15 - 18:25

Final Score: 2/10

Weak Topics: ['Neural Networks', 'Neural Network Architecture', 'Activation Functions', 'Machine Learning', 'Linear Algebra', 'Image Recognition', 'Feature Extraction', 'Training']


In [8]:
def run_pipeline(url, num_questions=10):
    # Agent 1
    agent1 = TranscriptionAgent()
    chunks = agent1.run(url)
    if not chunks:
        return None
    print(f"Agent 1 done: {len(chunks)} chunks")

    # Agent 1.5: Filter
    agent_filter = FilterAgent(llm)
    filtered_chunks = agent_filter.run(chunks)
    if not filtered_chunks:
        return None
    
    # Select chunks from filtered ones
    total_chunks = len(filtered_chunks)
    if total_chunks <= num_questions:
        selected_chunks = filtered_chunks
    else:
        step = total_chunks // num_questions
        selected_chunks = [filtered_chunks[i * step] for i in range(num_questions)]
    print(f"Selected {len(selected_chunks)} chunks for quiz")
    
    # Agent 2: Extract concepts
    agent2 = ConceptExtractionAgent(llm)
    concepts = agent2.run(selected_chunks)
    if not concepts:
        return None
    else:
        print(f"Agent 2 done: {len(concepts)} concept sets")
     
    # Agent 3
    agent3 = QuestionGenerationAgent(llm)
    quiz = agent3.run(concepts)
    if not quiz:
        return None
    else:
        print(f"Agent 3 done: {len(quiz['questions'])} questions")
    
    # Agent 4
    agent4 = EvaluationAgent(llm)
    results = agent4.run(quiz)

    # Save results
    # Save results
    tracker = QuizTracker()
    all_topics = []
    for q in quiz["questions"]:
        all_topics.extend(q.get("concepts", []))
    all_topics = list(set(all_topics))
    tracker.save_result(url, results["score"], results["total"], results["weak_topics"], all_topics)

    # Store filtered chunks in results
    results["filtered_chunks"] = filtered_chunks

    # Ask if user wants adaptive quiz
    if results["weak_topics"]:
        take_adaptive = input("\nYou have weak topics. Want to take an adaptive quiz? (yes/no): ").lower()
        if take_adaptive == "yes":
            adaptive_results = run_adaptive_quiz(results["weak_topics"], filtered_chunks)
            results["adaptive_results"] = adaptive_results
    
    return results

In [9]:
def run_adaptive_quiz(weak_topics, filtered_chunks):
    matching_chunks = []
    
    for chunk in filtered_chunks:
        prompt = f"""Here are the topics the student is weak at:
{weak_topics}
Does this transcript section cover any of these topics?
Transcript:
{chunk['text']}
Reply with ONLY one word: "yes" or "no"
"""
        try:
            response = llm.invoke(prompt)
            content = response.content.strip().lower()
            if content == "yes":
                matching_chunks.append(chunk)
        except Exception as e:
            print(f"Error matching chunk: {e}")
            continue
    
    print(f"Found {len(matching_chunks)} chunks matching weak topics")

    # Limit to 5 chunks to save API calls
    if len(matching_chunks) > 5:
        step = len(matching_chunks) // 5
        matching_chunks = [matching_chunks[i * step] for i in range(5)]
        print(f"Selected 5 chunks for adaptive quiz")
    
    if not matching_chunks:
        print("No matching chunks found")
        return None
    
    # Agent 2
    agent2 = ConceptExtractionAgent(llm)
    concepts = agent2.run(matching_chunks)
    if not concepts:
        return None
    print(f"Agent 2 done: {len(concepts)} concept sets")
    
    # Agent 3
    agent3 = QuestionGenerationAgent(llm)
    quiz = agent3.run(concepts)
    if not quiz:
        return None
    print(f"Agent 3 done: {len(quiz['questions'])} questions")
    
    # Agent 4
    agent4 = EvaluationAgent(llm)
    results = agent4.run(quiz)
    
    return results

In [11]:
from datetime import datetime
class QuizTracker:
    def __init__(self, filename="quiz_history.json"):
        self.filename = filename
        if os.path.exists(self.filename):
            with open(self.filename, "r") as f:
                self.history = json.load(f)
        else:
            self.history = []
    
    def save_result(self, url, score, total, weak_topics, all_topics):
        result = {
            "url": url,
            "score": score,
            "total": total,
            "weak_topics": weak_topics,
            "all_topics": all_topics,
            "date": datetime.now().strftime("%Y-%m-%d %H:%M")
        }
        self.history.append(result)
        with open(self.filename, "w") as f:
            json.dump(self.history, f, indent=2)
        print(f"Result saved! Total quizzes taken: {len(self.history)}")
    
    def get_weak_topics(self):
        all_weak = []
        for entry in self.history:
            all_weak.extend(entry["weak_topics"])
        return list(set(all_weak))
    
    def get_history(self):
        for entry in self.history:
            print(f"\nDate: {entry['date']}")
            print(f"Video: {entry['url']}")
            print(f"Score: {entry['score']}/{entry['total']}")
            print(f"Weak Topics: {entry['weak_topics']}")
        return self.history
    
    def get_progress(self):
        topic_stats = {}
        
        for entry in self.history:
            for topic in entry.get("all_topics", []):
                if topic not in topic_stats:
                    topic_stats[topic] = {"attempted": 0, "failed": 0}
                topic_stats[topic]["attempted"] += 1
                if topic in entry["weak_topics"]:
                    topic_stats[topic]["failed"] += 1
        
        print("\n--- Progress Summary ---")
        for topic, stats in topic_stats.items():
            passed = stats["attempted"] - stats["failed"]
            if stats["failed"] == 0:
                status = "Mastered"
            elif stats["failed"] > passed:
                status = "Needs work"
            else:
                status = "Improving"
            print(f"{topic}: Attempted {stats['attempted']} | Passed {passed} | Failed {stats['failed']} | {status}")
        
        return topic_stats

In [12]:
#results = run_pipeline("https://www.youtube.com/watch?v=_gGRBWaRpAQ&list=PLZoTAELRMXVNNrHSKv36Lr3_156yCo6Nn&index=5")

# Round 1
results = run_pipeline("https://www.youtube.com/watch?v=_gGRBWaRpAQ&list=PLZoTAELRMXVNNrHSKv36Lr3_156yCo6Nn&index=5")

# Round 2
adaptive_results = run_adaptive_quiz(results["weak_topics"], results["filtered_chunks"])

Agent 1 done: 37 chunks
Skipping filler chunk at 14s
Skipping filler chunk at 134s
Skipping filler chunk at 254s
Skipping filler chunk at 3295s
Skipping filler chunk at 3415s
Skipping filler chunk at 3537s
Skipping filler chunk at 3780s
Skipping filler chunk at 4146s
Skipping filler chunk at 4267s
Skipping filler chunk at 4391s
Kept 27 out of 37 chunks
Selected 10 chunks for quiz
Agent 2 done: 10 concept sets
Agent 3 done: 10 questions

Question 1: What is the primary purpose of removing stop words during text preprocessing in a Bag of Words model?
  A) To remove words that appear infrequently in the corpus
  B) To normalize the text data by converting all words to lowercase
  C) To reduce the dimensionality of the feature space by removing rare words
  D) To eliminate common words that do not carry significant meaning in the text, such as 'the' and 'and'



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: D
Explanation: Stop words are common words like 'the', 'and', 'a', etc. that do not add much value to the meaning of the text. Removing them helps reduce noise and improves the model's performance by focusing on more meaningful words. This is a key step in text preprocessing for Bag of Words models.
Review in video at: 6:21 - 8:23

Question 2: When dealing with out-of-vocabulary words in a natural language processing model that uses vector representation, which of the following is a common approach to handle the issue while considering the storage efficiency of a sparse matrix?
  A) Increase the dimensionality of the vector representation to accommodate more words
  B) Use a subword modeling technique to represent the out-of-vocabulary words
  C) Store all possible word vectors in a dense matrix
  D) Remove the out-of-vocabulary words from the dataset



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: This is correct because subword modeling techniques, such as WordPiece or BPE, can effectively represent out-of-vocabulary words by breaking them down into subwords that are present in the training vocabulary. This approach allows the model to generate vector representations for unseen words, improving its ability to handle out-of-vocabulary words. Additionally, using a sparse matrix to store the vector representations can efficiently store the sparse data, reducing storage requirements.
Review in video at: 10:24 - 12:26

Question 3: What is the primary purpose of using TF-IDF in conjunction with cosine similarity when analyzing the semantic meaning of text documents?
  A) To increase the weight of rare words and decrease the weight of common words, facilitating more accurate cosine similarity calculations
  B) To decrease the weight of rare words and increase the weight of common words, making the text analysis more dependent on frequent terms

...


Your answer (A/B/C/D):  a


Correct!
Correct Answer: A
Explanation: Lemmatization is the correct answer because it reduces words to their base or root form, known as the lemma, while also considering the context and grammar of the language. This results in a more accurate representation of the word's meaning, as opposed to stemming, which simply removes suffixes without considering the context. Stopwords removal removes common words like 'the' and 'and', while tokenization is the process of breaking text into individual words or tokens.
Review in video at: 38:44 - 40:44

Question 10: In a text processing task using N-grams, which of the following is a common issue that arises due to the large number of unique words or phrases in the dataset?
  A) The sparsity problem, where most N-gram frequencies are zero
  B) Overfitting due to too many features
  C) The curse of dimensionality, where most features are irrelevant
  D) Underfitting due to too few features



Your answer (A/B/C/D):  a


Correct!
Correct Answer: A
Explanation: The sparsity problem is a common issue in N-gram text processing, where most N-gram frequencies are zero, resulting in a sparse matrix. This occurs because the number of possible N-grams grows exponentially with the size of the vocabulary and the length of the N-grams, making it likely that most N-grams will not appear in the dataset. This can lead to difficulties in modeling and analyzing the data.
Review in video at: 42:47 - 44:48

Final Score: 5/10

Weak Topics: ['Natural Language Processing (NLP)', 'Text Preprocessing', 'Vector Representation', 'Word Importance', 'Bag of Words and Term Frequency-Inverse Document Frequency (TF-IDF)']


TypeError: QuizTracker.save_result() missing 1 required positional argument: 'all_topics'

In [ ]:
# Delete old history file to start fresh
import os
if os.path.exists("quiz_history.json"):
    os.remove("quiz_history.json")

tracker = QuizTracker()

# Quiz 1
tracker.save_result(
    url="https://youtube.com/video1",
    score=6,
    total=10,
    weak_topics=["TF-IDF", "Cosine Similarity", "N-grams"],
    all_topics=["TF-IDF", "Cosine Similarity", "N-grams", "Stemming", "Bag of Words"]
)

# Quiz 2
tracker.save_result(
    url="https://youtube.com/video2",
    score=7,
    total=10,
    weak_topics=["TF-IDF", "N-grams"],
    all_topics=["TF-IDF", "Cosine Similarity", "N-grams", "Stemming", "Lemmatization"]
)

# Quiz 3
tracker.save_result(
    url="https://youtube.com/video3",
    score=8,
    total=10,
    weak_topics=["TF-IDF"],
    all_topics=["TF-IDF", "Cosine Similarity", "Bag of Words", "Stemming"]
)

tracker.get_progress()